# AME-TS evaluation

This notebook builds a comparison table from GIFT-Eval result CSVs and visualizes router embeddings.
Run it from the `evaluation/` directory.

## Comparison table

In [ ]:
from eval_utils import generate_comprehensive_comparison_table

In [ ]:
# CSV with one selected config per task, for example test_valbest.csv from
# scripts/evaluate_protocol.sh. Set RUN to your results directory.
RUN = "../results_combined/<run>"

table = generate_comprehensive_comparison_table(
    ame_results_path=f"{RUN}/test_valbest.csv",
    baseline_paths={
        "Baseline": "<path/to/baseline/all_results.csv>",
    },
    improvement_baseline="Baseline",
    metrics=["eval_metrics/MASE[0.5]", "eval_metrics/RMSE[mean]",
             "eval_metrics/sMAPE[0.5]", "eval_metrics/MAE[0.5]"],
)

## Router embeddings

`extract_router_embeddings.py` runs a checkpoint over GIFT-Eval test inputs and saves per-layer router logits,
encoder representations, expert assignments, and descriptor predictions. Pass `--save_prefix moe` for a
standard MoE baseline checkpoint so that the visualization labels it as such.

In [ ]:
%%bash
CUDA_VISIBLE_DEVICES=0 python extract_router_embeddings.py \
    --dataset m4_monthly,lotsa_m5,lotsa_traffic_hourly,m4_yearly \
    --checkpoint <path/to/checkpoint.ckpt> \
    --max_context 512 \
    --max_samples 5000

In [ ]:
import warnings
import os
warnings.filterwarnings("ignore")
from eval_utils import visualize_adaptive_multiregime

# Directory name written by the extraction step.
dataset = "multi_m4_monthly_lotsa_m5_lotsa_traffic_hourly_+1more"
vis_features = ["forecastability", "sparsity"]
vis_str = "_".join(vis_features)
os.makedirs("figures", exist_ok=True)

for layer in range(4):
    visualize_adaptive_multiregime(
        embeddings_dir="router_embeddings",
        model_type="amets",
        dataset=dataset,
        layer=layer,
        n_samples=15000,
        method="tsne",
        balance_classes=True,
        show_encoder_plot=True,
        vis_features=vis_features,
        save_path=f"figures/amets_multi_dataset_{layer}_{vis_str}_tsne.png",
    )